# 01 — FMAC cleaning

Cleans and encodes the two Florida origination cohorts, then writes the artifacts every later FMAC stage reads and none recomputes. The split is the calendar: the 2023 cohort is the development partition, the 2024 cohort is the test partition.

## Outputs

- `data/fmac_train_processed.parquet` — the 2023 cohort, encoded (the tabular view)
- `data/fmac_test_processed.parquet` — the 2024 cohort, encoded on the 2023 level set
- `data/fmac_processed.parquet` — both cohorts pooled, one row per retained loan; the row position **is** the `borrower_id` every graph index is built from
- `outputs/fmac_frozen_split.json` — the temporal split, its counts and its checksums
- `outputs/fmac_bootstrap_indices.npy` — the one shared, outcome-stratified resample matrix over the test partition

Named steps live in `fmac/cleaning.py`; what the dataset *is* (layers, networks, split, artifact paths) is `fmac/config.py`. Every run parameter comes from `config.py`; none is defined here.

## The two network layers

| Layer | Raw column | States fitted on 2023 |
|---|---|---:|
| 1 | `PostalCode` | 24 |
| 2 | `SellerName` | 19 |

`MSA` is dropped. A metropolitan-statistical-area code is determined by the geography `PostalCode` already carries, so keeping both puts the same relation into the frame twice and lets a second family of columns stand in for the layer-1 attribute.

`SellerName` is kept and encoded. It is the originator relation, and "loans sold by the same seller" is exactly the kind of shared recorded attribute a MuLP layer is.

## Raw-field decision specification

| Raw column | Action | Output | Rationale |
|---|---|---|---|
| `FICO` | Keep | Numeric | Valid observed credit scores |
| `FirstPaymentDate` | Drop | — | Constant within cohort and would identify the cohort year |
| `FirstTimeHomebuyerIndicator` | Encode | `FirstTimeHomebuyerIndicator_Y` | Binary `N/Y` field |
| `MaturityDate` | Drop | — | Redundant with first-payment date and loan term |
| `MSA` | Drop | — | Determined by the geography `PostalCode` carries; layer 1 is the ZIP prefix, and the same relation must not enter the frame twice |
| `MI%` | Keep | Numeric | Valid discrete mortgage-insurance percentage |
| `NumberOfUnits` | Keep | Numeric | Valid count from 1–4 |
| `OccupancyStatus` | Encode | `(encoded from OccupancyStatus)` | Nominal category |
| `CLTV` | Keep | Numeric | Valid leverage measure including secondary financing |
| `DTI` | Keep | Numeric | Valid debt-to-income percentage |
| `OriginalUPB` | Keep | Numeric | Valid rounded original balance |
| `LTV` | Keep | Numeric | Retained alongside CLTV by decision |
| `OriginalInterestRate` | Keep | Numeric | Valid original note rate |
| `Channel` | Encode | `(encoded from Channel)` | Nominal acquisition channel |
| `PrepaymentPenaltyIndicator` | Drop | — | Constant `N` for all rows |
| `AmortizationType` | Drop | — | Constant `FRM` for all rows |
| `PropertyState` | Drop | — | Constant `FL` for all rows |
| `PropertyType` | Encode | `(encoded from PropertyType)` | Nominal property category |
| `PostalCode` | Encode | `(encoded from PostalCode)` | **Layer 1.** Three-digit ZIP prefix, a nominal code and not a number |
| `LoanIdentifier` | Drop | — | Unique for every loan; dropped to prevent memorization |
| `LoanPurpose` | Encode | `(encoded from LoanPurpose)` | Nominal loan-purpose code |
| `OriginalLoanTerm` | Keep | Numeric | Valid scheduled-payment count |
| `NumberOfBorrowers` | Keep | Numeric | Valid borrower count |
| `SellerName` | Encode | `(encoded from SellerName)` | **Layer 2.** The originator relation |
| `SuperConformingFlag` | Drop | — | Only one `Y` in 2023 and none in 2024 |
| `Pre-HARPLoanSequence Number` | Drop | — | All loans in the two cohorts are non-HARP, so every entry is null |
| `SpecialEligibilityProgram` | Encode | `(encoded from SpecialEligibilityProgram)` | Preserves `F`, `H` and absence as separate states |
| `HARPIndicator` | Drop | — | Constant `N` for all rows |
| `PropertyValuationMethod` | Encode | `(encoded from PropertyValuationMethod)` | Nominal code, not an ordered number |
| `InterestOnly(I/O)Indicator` | Drop | — | Constant `N` for all rows |
| `VantageScore` | Drop | — | Constant `9999` for all rows |
| `target` | Keep as label | `target`, and `BAD` in the pooled frame | The binary outcome |

## Output schema

| Output column or family | Type | Count |
|---|---|---:|
| `FICO`, `MI%`, `NumberOfUnits`, `CLTV`, `DTI`, `OriginalUPB`, `LTV`, `OriginalInterestRate`, `OriginalLoanTerm`, `NumberOfBorrowers` | Numeric | 10 |
| `FirstTimeHomebuyerIndicator_Y` | 0/1 integer | 1 |
| `(encoded from PostalCode)` — layer 1 | 0/1 integers | 25 |
| `(encoded from SellerName)` — layer 2 | 0/1 integers | 20 |
| `(encoded from OccupancyStatus)` | 0/1 integers | 4 |
| `(encoded from Channel)` | 0/1 integers | 4 |
| `(encoded from PropertyType)` | 0/1 integers | 5 |
| `(encoded from LoanPurpose)` | 0/1 integers | 4 |
| `(encoded from PropertyValuationMethod)` | 0/1 integers | 4 |
| `(encoded from SpecialEligibilityProgram)` | 0/1 integers | 3 |
| `target` (`BAD` in the pooled frame) | 0/1 integer label | 1 |

80 predictors and the label. Each one-hot family reserves a `Missing` level, except `SpecialEligibilityProgram`, whose absent state already *is* that level.

## How levels, absence and the split are handled

**The level set is fitted on 2023 alone.** The test cohort is encoded against it, so a postal prefix or a seller that appears only in 2024 is given no category node. How many test rows that affects is printed below and recorded in the split file: it is measured every run, not assumed.

**Absence has one name, and it is never a node.** Every family reserves the level `Missing` (`preprocessing.missing_levels.MISSING_SENTINEL`), used both for a genuinely absent value and for a level unseen in the development cohort. In the two layer families this is what keeps the graph honest: the repository identifies an absence marker by the exact `_Missing` suffix and refuses to give it a category node, because "this field was blank, or held something never seen before" is not an attribute borrowers can share. Linking every such borrower into one clique would put a hub in the middle of the graph that propagates default risk between borrowers with nothing in common.

**The numeric columns need no imputation.** They arrive complete in both cohorts, which the contract cells below assert rather than assume.

**The split is the calendar.** 2023 is development, 2024 is test. There is no seed and nothing is stratified, because nothing is drawn. Borrowers whose every layer field landed in the `Missing` level are dropped from the pooled frame, once, exactly as HMEQ drops its unconnectable rows: they touch no category node, so they carry no edge in any operator built from the frame.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def repo_root(start: Path | None = None) -> Path:
    """The qmulp checkout this notebook lives in."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("run this notebook from inside the qmulp checkout")


ROOT = repo_root()
sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from config import DEFAULT_CONFIG
from evaluation.provenance import RunTimer
from fingerprint import resolve_config, sha256_path
from fmac.config import DATASET as dataset
from fmac.config import TEST_PROCESSED, TRAIN_PROCESSED, raw_csvs
from paths import repo_path
from preprocessing.missing_levels import MISSING_SENTINEL

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

config, smoke = resolve_config(DEFAULT_CONFIG)
timer = RunTimer()
RAW_CSVS = raw_csvs()
OUTPUT_PATHS = {
    "train": repo_path(TRAIN_PROCESSED),
    "test": repo_path(TEST_PROCESSED),
    "network": dataset.processed_path(),
    "split": dataset.frozen_split_path(),
    "bootstrap": dataset.bootstrap_path(),
}
print("repository:", ROOT)
print("dataset:", dataset.name, "| smoke:", smoke)
print("inputs:", {year: path.name for year, path in RAW_CSVS.items()})
print("outputs:", {key: path.relative_to(ROOT).as_posix() for key, path in OUTPUT_PATHS.items()})

In [ ]:
EXPECTED_RAW_COLUMNS = [
    "FICO",
    "FirstPaymentDate",
    "FirstTimeHomebuyerIndicator",
    "MaturityDate",
    "MSA",
    "MI%",
    "NumberOfUnits",
    "OccupancyStatus",
    "CLTV",
    "DTI",
    "OriginalUPB",
    "LTV",
    "OriginalInterestRate",
    "Channel",
    "PrepaymentPenaltyIndicator",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanIdentifier",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "SuperConformingFlag",
    "Pre-HARPLoanSequence Number",
    "SpecialEligibilityProgram",
    "HARPIndicator",
    "PropertyValuationMethod",
    "InterestOnly(I/O)Indicator",
    "VantageScore",
    "target",
]
NUMERIC_COLUMNS = [
    "FICO",
    "MI%",
    "NumberOfUnits",
    "CLTV",
    "DTI",
    "OriginalUPB",
    "LTV",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
]
BINARY_COLUMN = "FirstTimeHomebuyerIndicator"
# The two network layers, in block order, then the remaining nominal fields.
# Only the first two become graph layers; the rest stay ordinary covariates.
LAYER_COLUMNS = ["PostalCode", "SellerName"]
OHE_COLUMNS = [
    *LAYER_COLUMNS,
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "PropertyValuationMethod",
    "SpecialEligibilityProgram",
]
DROPPED_COLUMNS = [
    "FirstPaymentDate",
    "MaturityDate",
    "MSA",
    "PrepaymentPenaltyIndicator",
    "AmortizationType",
    "PropertyState",
    "LoanIdentifier",
    "SuperConformingFlag",
    "Pre-HARPLoanSequence Number",
    "HARPIndicator",
    "InterestOnly(I/O)Indicator",
    "VantageScore",
]
# Levels fitted on the 2023 cohort: written down where the domain is a closed
# code list, recorded as a count where the levels are names, so that no vendor
# list is carried in the source.
EXPECTED_TRAIN_CATEGORIES = {
    "PostalCode": [
        "321",
        "322",
        "323",
        "324",
        "325",
        "326",
        "327",
        "328",
        "329",
        "330",
        "331",
        "333",
        "334",
        "335",
        "336",
        "337",
        "338",
        "339",
        "341",
        "342",
        "344",
        "346",
        "347",
        "349",
    ],
    "OccupancyStatus": ["I", "P", "S"],
    "Channel": ["B", "C", "R"],
    "PropertyType": ["CO", "MH", "PU", "SF"],
    "LoanPurpose": ["C", "N", "P"],
    "PropertyValuationMethod": ["1", "2", "4"],
    "SpecialEligibilityProgram": ["F", "H", MISSING_SENTINEL],
}
EXPECTED_TRAIN_LEVEL_COUNTS = {"PostalCode": 24, "SellerName": 19}
# Predictors after encoding: 10 numeric + 1 binary + the one-hot families.
EXPECTED_N_FEATURES = 80
EXPECTED_ROWS = {"2023": 3045, "2024": 3199}
EXPECTED_TARGET_COUNTS = {"2023": {0: 2954, 1: 91}, "2024": {0: 3096, 1: 103}}
EXPECTED_NULL_COUNTS = {
    "2023": {"MSA": 35, "Pre-HARPLoanSequence Number": 3045, "SpecialEligibilityProgram": 2930},
    "2024": {"MSA": 28, "Pre-HARPLoanSequence Number": 3199, "SpecialEligibilityProgram": 3112},
}

In [ ]:
source_hashes_before = {year: sha256_path(path) for year, path in RAW_CSVS.items()}
raw = {year: pd.read_csv(path) for year, path in RAW_CSVS.items()}
raw_snapshots = {year: frame.copy(deep=True) for year, frame in raw.items()}

print({year: frame.shape for year, frame in raw.items()})
print("source SHA-256:", source_hashes_before)

In [ ]:
for year, frame in raw.items():
    assert frame.columns.tolist() == EXPECTED_RAW_COLUMNS
    assert len(frame) == EXPECTED_ROWS[year]
    assert frame["LoanIdentifier"].notna().all()
    assert frame["LoanIdentifier"].is_unique
    assert set(frame["target"].unique()) == {0, 1}
    assert frame["target"].value_counts().sort_index().to_dict() == EXPECTED_TARGET_COUNTS[year]
    observed_nulls = frame.isna().sum()
    observed_nulls = observed_nulls[observed_nulls.gt(0)].to_dict()
    assert observed_nulls == EXPECTED_NULL_COUNTS[year]
    parsed_payment = pd.to_datetime(frame["FirstPaymentDate"], errors="raise")
    assert parsed_payment.nunique() == 1
    assert parsed_payment.iloc[0] == pd.Timestamp(f"{year}-03-01")

assert not set(raw["2023"]["LoanIdentifier"]).intersection(raw["2024"]["LoanIdentifier"])
print("Raw contracts passed: schema, rows, IDs, targets, missingness, dates, cohort separation.")

In [ ]:
EXPECTED_CONSTANTS = {
    "PrepaymentPenaltyIndicator": "N",
    "AmortizationType": "FRM",
    "PropertyState": "FL",
    "HARPIndicator": "N",
    "InterestOnly(I/O)Indicator": "N",
    "VantageScore": 9999,
}
EXPECTED_DOMAINS = {
    "FirstTimeHomebuyerIndicator": {"N", "Y"},
    "OccupancyStatus": {"I", "P", "S"},
    "Channel": {"B", "C", "R"},
    "PropertyType": {"CO", "MH", "PU", "SF"},
    "LoanPurpose": {"C", "N", "P"},
    "SpecialEligibilityProgram": {"F", "H"},
    "PropertyValuationMethod": {1, 2, 4},
}

for year, frame in raw.items():
    assert not frame[NUMERIC_COLUMNS].isna().any().any()
    assert np.isfinite(frame[NUMERIC_COLUMNS].to_numpy(dtype=float)).all()
    for column, value in EXPECTED_CONSTANTS.items():
        assert frame[column].eq(value).all()
    for column, expected in EXPECTED_DOMAINS.items():
        assert set(frame[column].dropna().unique()) == expected
    # Both layer fields must be complete. An absent one would be a marker, and
    # the notebook would have to say so rather than encode it silently.
    for column in LAYER_COLUMNS:
        assert frame[column].notna().all(), column
    sellers = frame["SellerName"].astype(str)
    assert sellers.eq(sellers.str.strip()).all(), "SellerName carries padding"
    if year == "2023":
        assert frame["SuperConformingFlag"].value_counts().to_dict() == {"N": 3044, "Y": 1}
    else:
        assert frame["SuperConformingFlag"].eq("N").all()

print("Cleaning assumptions passed: numerics complete, domains unchanged, layer fields present.")

In [ ]:
def normalized_categories(frame):
    """Every encoded field as a string, with an absent value at the reserved level."""
    out = pd.DataFrame(index=frame.index)
    out["PostalCode"] = frame["PostalCode"].map(lambda value: f"{int(value):03d}")
    out["SellerName"] = frame["SellerName"].astype(str)
    out["OccupancyStatus"] = frame["OccupancyStatus"].astype(str)
    out["Channel"] = frame["Channel"].astype(str)
    out["PropertyType"] = frame["PropertyType"].astype(str)
    out["LoanPurpose"] = frame["LoanPurpose"].astype(str)
    out["PropertyValuationMethod"] = frame["PropertyValuationMethod"].map(
        lambda value: str(int(value))
    )
    out["SpecialEligibilityProgram"] = (
        frame["SpecialEligibilityProgram"].fillna(MISSING_SENTINEL).astype(str)
    )
    return out[OHE_COLUMNS]


normalized = {year: normalized_categories(frame) for year, frame in raw.items()}
for year, frame in raw.items():
    pd.testing.assert_frame_equal(frame, raw_snapshots[year])

display(normalized["2023"].head())

In [ ]:
from fmac.cleaning import sanitize_level, sanitized_levels

fitted_levels = {
    column: sorted(normalized["2023"][column].unique().tolist()) for column in OHE_COLUMNS
}
for column, expected in EXPECTED_TRAIN_CATEGORIES.items():
    assert fitted_levels[column] == sorted(expected), column
for column, expected_count in EXPECTED_TRAIN_LEVEL_COUNTS.items():
    assert len(fitted_levels[column]) == expected_count, (column, len(fitted_levels[column]))

# Every family reserves the absence level. SpecialEligibilityProgram already
# fitted it from its nulls, so it is not added a second time.
category_levels = {
    column: [
        *fitted_levels[column],
        *([] if MISSING_SENTINEL in fitted_levels[column] else [MISSING_SENTINEL]),
    ]
    for column in OHE_COLUMNS
}
level_columns = {
    column: {level: f"{column}_{sanitize_level(level)}" for level in category_levels[column]}
    for column in OHE_COLUMNS
}
for column in OHE_COLUMNS:
    sanitized_levels(category_levels[column])  # refuses a sanitisation that merges two levels

# How much of the test cohort the 2023 level set does not cover. Those rows
# lose the edge that level would have carried, so the count is reported.
unseen_in_development = {
    column: int((~normalized["2024"][column].isin(fitted_levels[column])).sum())
    for column in OHE_COLUMNS
}
display(
    pd.DataFrame(
        {
            "column": OHE_COLUMNS,
            "n_levels_fitted": [len(fitted_levels[c]) for c in OHE_COLUMNS],
            "n_columns": [len(category_levels[c]) for c in OHE_COLUMNS],
            "is_layer": [c in LAYER_COLUMNS for c in OHE_COLUMNS],
            "n_test_rows_at_unseen_level": [unseen_in_development[c] for c in OHE_COLUMNS],
        }
    )
)

In [ ]:
def transform(frame, categories):
    """Numerics, the binary flag and every one-hot family, on the fitted levels."""
    output = frame[NUMERIC_COLUMNS].copy()
    output["FirstTimeHomebuyerIndicator_Y"] = (
        frame[BINARY_COLUMN].map({"N": 0, "Y": 1}).astype("int8")
    )
    family_columns = {}
    for column in OHE_COLUMNS:
        source = categories[column]
        mapped = source.where(source.isin(fitted_levels[column]), MISSING_SENTINEL)
        columns = []
        for level in category_levels[column]:
            name = level_columns[column][level]
            output[name] = mapped.eq(level).astype("int8")
            columns.append(name)
        family_columns[column] = columns
    output["target"] = frame["target"].astype("int8")
    return output, family_columns


processed = {}
for year in ("2023", "2024"):
    processed[year], encoded_families = transform(raw[year], normalized[year])

print({year: frame.shape for year, frame in processed.items()})
print("family widths:", {key: len(value) for key, value in encoded_families.items()})

In [ ]:
feature_columns = [
    *NUMERIC_COLUMNS,
    "FirstTimeHomebuyerIndicator_Y",
    *(column for family in OHE_COLUMNS for column in encoded_families[family]),
]
output_columns = [*feature_columns, "target"]
assert len(feature_columns) == EXPECTED_N_FEATURES, len(feature_columns)
assert len(set(output_columns)) == len(output_columns), "duplicate output column name"

train_processed = processed["2023"].reindex(columns=output_columns)
test_processed = processed["2024"].reindex(columns=output_columns)
X_train = train_processed[feature_columns].copy()
y_train = train_processed["target"].copy()
X_test = test_processed[feature_columns].copy()
y_test = test_processed["target"].copy()

print("X_train/y_train:", X_train.shape, y_train.shape)
print("X_test/y_test:", X_test.shape, y_test.shape)

In [ ]:
assert train_processed.shape == (EXPECTED_ROWS["2023"], EXPECTED_N_FEATURES + 1)
assert test_processed.shape == (EXPECTED_ROWS["2024"], EXPECTED_N_FEATURES + 1)
assert train_processed.columns.tolist() == test_processed.columns.tolist() == output_columns
assert train_processed.columns[-1] == "target"
assert not train_processed.isna().any().any()
assert not test_processed.isna().any().any()
assert not train_processed.select_dtypes(include=["object", "string"]).columns.tolist()
assert not test_processed.select_dtypes(include=["object", "string"]).columns.tolist()
assert np.isfinite(train_processed.to_numpy(dtype=float)).all()
assert np.isfinite(test_processed.to_numpy(dtype=float)).all()
assert y_train.value_counts().sort_index().to_dict() == EXPECTED_TARGET_COUNTS["2023"]
assert y_test.value_counts().sort_index().to_dict() == EXPECTED_TARGET_COUNTS["2024"]

for family, columns in encoded_families.items():
    for frame in (train_processed, test_processed):
        assert set(np.unique(frame[columns].to_numpy())).issubset({0, 1})
        assert frame[columns].sum(axis=1).eq(1).all(), family

# In a layer family exactly one column may look like an absence marker, and it
# must carry the exact suffix the repository reads.
for layer in LAYER_COLUMNS:
    markers = [c for c in encoded_families[layer] if c.endswith(f"_{MISSING_SENTINEL}")]
    assert markers == [f"{layer}_{MISSING_SENTINEL}"], (layer, markers)

assert set(np.unique(train_processed["FirstTimeHomebuyerIndicator_Y"])).issubset({0, 1})
assert set(np.unique(test_processed["FirstTimeHomebuyerIndicator_Y"])).issubset({0, 1})
assert not set(DROPPED_COLUMNS).intersection(output_columns)
assert not [c for c in output_columns if c.startswith("MSA")], "MSA survived the drop"
for column in NUMERIC_COLUMNS:
    pd.testing.assert_series_equal(train_processed[column], raw["2023"][column], check_names=True)
    pd.testing.assert_series_equal(test_processed[column], raw["2024"][column], check_names=True)
pd.testing.assert_series_equal(y_train.astype("int64"), raw["2023"]["target"], check_names=True)
pd.testing.assert_series_equal(y_test.astype("int64"), raw["2024"]["target"], check_names=True)
for year, frame in raw.items():
    pd.testing.assert_frame_equal(frame, raw_snapshots[year])

print("Output validation passed: shape, schema, nulls, one-hot sums, layer markers, row order.")

In [ ]:
for path in OUTPUT_PATHS.values():
    path.parent.mkdir(parents=True, exist_ok=True)

pl.from_pandas(train_processed).write_parquet(OUTPUT_PATHS["train"])
pl.from_pandas(test_processed).write_parquet(OUTPUT_PATHS["test"])
reloaded_train = pd.read_parquet(OUTPUT_PATHS["train"])
reloaded_test = pd.read_parquet(OUTPUT_PATHS["test"])
pd.testing.assert_frame_equal(reloaded_train, train_processed)
pd.testing.assert_frame_equal(reloaded_test, test_processed)
assert {year: sha256_path(path) for year, path in RAW_CSVS.items()} == source_hashes_before
print("per-cohort frames written:", reloaded_train.shape, reloaded_test.shape)
print("dropped raw columns:", DROPPED_COLUMNS)

## The pooled network frame and the frozen split

Both cohorts in one frame, 2023 first: the row position is the `borrower_id` every graph index is built from, so the two partitions are contiguous ranges. The outcome is renamed `BAD`, the name the operator reads. Borrowers with no genuine category in either layer are dropped here, once, and recorded in the split file.

In [ ]:
from fmac import cleaning as fmac
from preprocessing.common import write_frozen_split

pooled, development_idx, test_idx = fmac.pool_cohorts(train_processed, test_processed)
pooled, development_idx, test_idx, dropped = fmac.drop_unconnectable(
    pooled, development_idx, test_idx
)
network_path, network_sha256 = fmac.write_network_frame(pooled, OUTPUT_PATHS["network"])
split_record = fmac.frozen_split_record(
    pooled,
    development_idx,
    test_idx,
    processed_path=network_path,
    processed_sha256=network_sha256,
    extra={
        "raw_sha256": source_hashes_before,
        "n_raw_rows": {year: int(len(frame)) for year, frame in raw.items()},
        "n_unconnectable_dropped": len(dropped),
        "dropped_pooled_positions": dropped,
        "levels_fitted_on": "2023",
        "layer_states": {column: len(fitted_levels[column]) for column in LAYER_COLUMNS},
        "n_test_rows_at_unseen_level": unseen_in_development,
    },
)
split_path = write_frozen_split(split_record, OUTPUT_PATHS["split"])
print("pooled frame:", pooled.shape)
print("unconnectable dropped:", len(dropped))
print("development / test rows:", development_idx.size, test_idx.size)
print("processed sha256:", network_sha256)

In [ ]:
from stages import data

# Everything after the cleaning is the shared data stage: it re-reads the frozen
# split (row count, per-partition default counts, processed checksum), draws the
# shared bootstrap matrix, writes the observation flags and Table 1's inputs.
result = data.run(dataset.name, config)
display(result["rows"].to_pandas())
print("differences from the expected counts:", result["notes"] or "none")
for path in result["artifacts"]:
    print(f"{sha256_path(path)}  {Path(path).resolve().relative_to(ROOT)}")
